# 03 · Feature Engineering & Preprocessing Pipeline

Following Géron Ch. 2 — *"Prepare the Data for Machine Learning Algorithms"*.

> *"Instead of doing this manually, you should write functions to do it for you, for several good reasons: … it allows you to easily prepare the data in any future project."* — Géron

We implement `RossmannFeatureTransformer` — a proper Scikit-Learn transformer that  
fits into a `Pipeline`, exactly as described in Chapter 2.

Key transformations:
1. Date parsing → year, month, day, week
2. **Cyclical encoding** (sin/cos) to preserve the circular nature of periodic features
3. Competition and promo duration in months / weeks
4. Imputation of missing values
5. Categorical label mapping

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120})

from rossmann_store_sales.config import load_config
from rossmann_store_sales.data import load_training_frame
from rossmann_store_sales.features import (
    prepare_features,
    RossmannFeatureTransformer,
    FEATURE_COLUMNS,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
)

cfg = load_config(ROOT / 'configs' / 'project.toml')
raw = load_training_frame(cfg).head(2000)
print(f'Raw sample: {raw.shape}')

## 1. The custom Scikit-Learn transformer

`RossmannFeatureTransformer` extends `BaseEstimator` and `TransformerMixin`,  
giving it `fit()` / `transform()` / `fit_transform()` for free — exactly the pattern  
taught in the book.

In [ ]:
# Works as a standalone transformer
transformer = RossmannFeatureTransformer(training=True)
prepared    = transformer.fit_transform(raw)

print('Input shape :', raw.shape)
print('Output shape:', prepared.shape)
print('\nNew columns added by the transformer:')
print(sorted(set(prepared.columns) - set(raw.columns)))

## 2. Feature list fed to the model

In [ ]:
print(f'Total features : {len(FEATURE_COLUMNS)}')
print(f'Categorical    : {CATEGORICAL_FEATURES}')
print(f'Numeric        : {len(NUMERIC_FEATURES)} columns')

prepared[FEATURE_COLUMNS].head()

## 3. Cyclical encoding — sin/cos explanation

A naive `day_of_week = 7` (Sunday) is far from `day_of_week = 1` (Monday) on a linear scale,  
but they are actually adjacent in the real world.  
Sine/cosine encoding places them correctly in a 2D circular space.

In [ ]:
days = np.arange(1, 8)
sin_vals = np.sin(days * (2 * np.pi / 7))
cos_vals = np.cos(days * (2 * np.pi / 7))
labels   = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(days, sin_vals, 'o-', label='sin', color='steelblue')
axes[0].plot(days, cos_vals, 's--', label='cos', color='darkorange')
axes[0].set_xticks(days); axes[0].set_xticklabels(labels)
axes[0].set_title('Sin/cos encoding — day of week')
axes[0].legend()

axes[1].scatter(sin_vals, cos_vals, c=days, cmap='hsv', s=120, zorder=5)
for i, lbl in enumerate(labels):
    axes[1].annotate(lbl, (sin_vals[i] + 0.04, cos_vals[i] + 0.04), fontsize=9)
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--')
axes[1].add_patch(circle)
axes[1].set_aspect('equal'); axes[1].set_title('Circular embedding')

plt.tight_layout()
plt.savefig(ROOT / 'reports/figures/cyclical_encoding.png', bbox_inches='tight')
plt.show()
print('Monday and Sunday are adjacent on the circle — as they should be.')

## 4. Full Scikit-Learn Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor

# The same pipeline used in production (rossmann_store_sales/models.py)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
        ('num', StandardScaler(), NUMERIC_FEATURES),
    ],
    remainder='drop',
    sparse_threshold=0.0,
)

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model',      DummyRegressor(strategy='mean')),   # placeholder
])

from rossmann_store_sales.features import model_frame

X_sample, y_sample, _ = model_frame(raw, training=True)
pipeline.fit(X_sample, np.log1p(y_sample))

print('Pipeline fitted. Steps:')
for name, step in pipeline.steps:
    print(f'  {name}: {step.__class__.__name__}')
print(f'\nTransformed X shape: {preprocessor.transform(X_sample).shape}')

## 5. Feature summary table

In [ ]:
summary = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'type': ['categorical' if f in CATEGORICAL_FEATURES else 'numeric' for f in FEATURE_COLUMNS],
    'encoding': [
        'OneHotEncoder' if f in CATEGORICAL_FEATURES
        else ('sin+cos' if f.endswith('_sin') or f.endswith('_cos') else 'StandardScaler')
        for f in FEATURE_COLUMNS
    ],
})
summary